# A/B Test Analysis: Conversion Rate Comparison  
### A Production-Ready Statistical Evaluation

**Author:** Folasunkanmi  
**Date:** November 2026  
**Repository:** [github.com/folasunkanmi/python-ab-testing](https://github.com/folasunkanmi/python-ab-testing)

---

### Abstract

This notebook presents a complete, reproducible A/B test analysis comparing two product versions (A and B) on a binary conversion outcome. We perform rigorous data validation (with early stopping on invalid records), a **sample-ratio mismatch (SRM) check**, estimate conversion rates with **95% Wilson confidence intervals**, conduct a **two-proportion z-test** as the primary hypothesis test, quantify practical significance, and perform **power and sample-size analysis**. The final section translates statistical findings into clear business recommendations.

**Key finding:** Version B showed a higher observed conversion rate (17.6% vs 14.0%), but the difference is **not statistically significant** at α = 0.05 (p ≈ 0.27). The experiment was underpowered for the observed effect size.

> **Data provenance note:** The dataset (`ab_data.csv`) was provided as an exercise file. Its original source, collection method, and any filtering steps are **unknown**. Treat results as illustrative rather than as evidence about a specific real product.


## 0. Setup & Reproducibility

All analysis uses open-source Python libraries. Results are fully reproducible with the provided dataset and environment specifications.


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from statsmodels.stats.proportion import (
    proportions_ztest,
    proportion_confint,
    confint_proportions_2indep
)
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

pd.set_option("display.float_format", "{:.4f}".format)
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("colorblind")

print("Environment ready.")
print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")


## 1. Data Loading & Strict Validation

A production-grade analysis **stops** when data quality problems are found. We check missing values, duplicates, invalid records, and group balance. Any failure raises an error and aborts the analysis.


In [ ]:
df = pd.read_csv("ab_data.csv")

print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)

errors = []

print(f"\n1. Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\n2. Schema:")
print(df.dtypes)
expected_cols = {"user_id", "group", "converted"}
if set(df.columns) != expected_cols:
    errors.append(f"Unexpected columns. Expected {expected_cols}, got {set(df.columns)}")

missing = df.isnull().sum()
print("\n3. Missing values:")
print(missing)
if missing.sum() > 0:
    errors.append(f"Missing values found: {missing[missing > 0].to_dict()}")

n_dupes = df.duplicated().sum()
n_user_dupes = df.duplicated(subset=["user_id"]).sum()
print(f"\n4. Duplicate rows: {n_dupes}")
print(f"   Duplicate user_ids: {n_user_dupes}")
if n_dupes > 0 or n_user_dupes > 0:
    errors.append(f"Duplicates found: {n_dupes} full-row, {n_user_dupes} user_id")

print("\n5. Value checks:")
print(f"   Groups present: {sorted(df['group'].unique())}")
print(f"   Converted values: {sorted(df['converted'].unique())}")
if not set(df["group"].unique()).issubset({"A", "B"}):
    errors.append(f"Invalid group labels: {df['group'].unique()}")
invalid_converted = ~df["converted"].isin([0, 1])
print(f"   Invalid 'converted' values: {invalid_converted.sum()}")
if invalid_converted.any():
    errors.append(f"Invalid converted values at indices: {df.index[invalid_converted].tolist()[:10]}")

print("\n6. Group counts:")
print(df["group"].value_counts().sort_index())

if errors:
    print("\n" + "!" * 60)
    print("VALIDATION FAILED — analysis stopped.")
    for e in errors:
        print(f"  • {e}")
    print("!" * 60)
    raise ValueError("Data validation failed. Fix the issues above before continuing.")
else:
    df_clean = df.copy()
    print("\n" + "=" * 60)
    print("VALIDATION PASSED — Dataset is clean and ready for analysis.")
    print("=" * 60)


## 1b. Sample Ratio Mismatch (SRM) Test

An SRM check verifies that the observed traffic split matches the intended allocation (here 50/50). A significant SRM can indicate a bug in assignment, logging, or filtering and can invalidate the experiment.


In [ ]:
observed = df_clean["group"].value_counts().sort_index()
n_total = observed.sum()
expected = [n_total / 2, n_total / 2]

chi2, srm_p = stats.chisquare(f_obs=observed.values, f_exp=expected)

print("=" * 60)
print("SAMPLE RATIO MISMATCH (SRM) TEST")
print("=" * 60)
print(f"Observed counts : A = {observed.get('A', 0)}, B = {observed.get('B', 0)}")
print(f"Expected counts : A = {expected[0]:.1f}, B = {expected[1]:.1f}")
print(f"Chi-square      : {chi2:.4f}")
print(f"p-value         : {srm_p:.4f}")
print()

srm_alpha = 0.001
if srm_p < srm_alpha:
    print(f"⚠️  SRM DETECTED (p < {srm_alpha}). Investigate assignment or logging before trusting results.")
else:
    print(f"✓ No significant SRM (p ≥ {srm_alpha}). Traffic split is consistent with 50/50 allocation.")


## 2. Conversion Rates with Confidence Intervals

We estimate the conversion rate for each variant and attach **95% Wilson confidence intervals**.


In [ ]:
def summarize_group(data, group_name):
    n = len(data)
    successes = int(data.sum())
    rate = successes / n
    ci_low, ci_high = proportion_confint(successes, n, alpha=0.05, method="wilson")
    return {"group": group_name, "conversions": successes, "users": n,
            "conversion_rate": rate, "ci_low": ci_low, "ci_high": ci_high}

a_data = df_clean.loc[df_clean["group"] == "A", "converted"]
b_data = df_clean.loc[df_clean["group"] == "B", "converted"]

summary_a = summarize_group(a_data, "A")
summary_b = summarize_group(b_data, "B")

summary_df = pd.DataFrame([summary_a, summary_b]).set_index("group")
summary_df["conversion_rate_%"] = summary_df["conversion_rate"] * 100
summary_df["ci_low_%"] = summary_df["ci_low"] * 100
summary_df["ci_high_%"] = summary_df["ci_high"] * 100

print("Conversion Rate Summary with 95% Confidence Intervals")
print("-" * 70)
display(summary_df[["users", "conversions", "conversion_rate_%", "ci_low_%", "ci_high_%"]])

rate_a = summary_a["conversion_rate"]
rate_b = summary_b["conversion_rate"]
abs_diff = rate_b - rate_a
rel_lift = abs_diff / rate_a

print(f"\nAbsolute difference (B − A): {abs_diff:.1%} points")
print(f"Relative lift of B over A:  {rel_lift:.1%}")


## 3. Hypothesis Testing

### Primary Test: Two-Proportion Z-Test

**Hypotheses**
- $H_0: p_A = p_B$
- $H_1: p_A \neq p_B$ (two-sided)


In [ ]:
count = np.array([summary_a["conversions"], summary_b["conversions"]])
nobs  = np.array([summary_a["users"], summary_b["users"]])

z_stat, p_value = proportions_ztest(count, nobs, alternative="two-sided")

ci_diff = confint_proportions_2indep(
    count[1], nobs[1], count[0], nobs[0],
    compare="diff", method="wald", alpha=0.05
)

print("=" * 60)
print("PRIMARY TEST: Two-Proportion Z-Test")
print("=" * 60)
print(f"z-statistic          : {z_stat: .4f}")
print(f"p-value              : {p_value: .4f}")
print(f"95% CI for (p_B - p_A): [{ci_diff[0]:.4f}, {ci_diff[1]:.4f}]")
print()

alpha = 0.05
if p_value < alpha:
    print(f"Result: Reject H0 at alpha = {alpha}. Difference is statistically significant.")
else:
    print(f"Result: Fail to reject H0 at alpha = {alpha}. Difference is NOT statistically significant.")

t_stat, t_p = stats.ttest_ind(a_data, b_data, equal_var=True)
print("\nSecondary check (independent t-test):")
print(f"t-statistic : {t_stat:.4f} | p-value: {t_p:.4f}")


## 4. Effect Size

Cohen’s h for the difference in proportions.


In [ ]:
h = proportion_effectsize(rate_b, rate_a)
print(f"Cohen's h (effect size): {h:.4f}")
print("\nRule of thumb for |h|: 0.20 small · 0.50 medium · 0.80 large")
print(f"Observed effect is {'small' if abs(h) < 0.5 else 'medium/large'}.")


## 5. Power Analysis & Required Sample Size


In [ ]:
analysis = NormalIndPower()
effect_size = proportion_effectsize(rate_b, rate_a)
nobs1 = summary_a["users"]

achieved_power = analysis.power(
    effect_size=effect_size, nobs1=nobs1, alpha=0.05, ratio=1.0, alternative="two-sided"
)

print("=" * 60)
print("POWER ANALYSIS")
print("=" * 60)
print(f"Observed effect size (Cohen's h): {effect_size:.4f}")
print(f"Sample size per group:            {nobs1}")
print(f"Achieved power:                   {achieved_power:.1%}")
print()

print("Required sample size per group for 80% power (alpha = 0.05, two-sided):")
print("-" * 60)
baseline = rate_a
for rel_lift_pct in [5, 10, 15, 20, 25]:
    target_rate = baseline * (1 + rel_lift_pct / 100)
    es = proportion_effectsize(target_rate, baseline)
    required_n = analysis.solve_power(effect_size=es, power=0.80, alpha=0.05, ratio=1.0, alternative="two-sided")
    print(f"  Detect {rel_lift_pct:2d}% relative lift → {int(np.ceil(required_n)):,} users per group")

print("\nInterpretation: Current experiment (n = 250/arm) has low power for the observed lift.")


## 6. Visualizations with Confidence Intervals


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax1 = axes[0]
groups = ["A", "B"]
rates = [rate_a * 100, rate_b * 100]
ci_lows = [summary_a["ci_low"] * 100, summary_b["ci_low"] * 100]
ci_highs = [summary_a["ci_high"] * 100, summary_b["ci_high"] * 100]
yerr = [[rates[i] - ci_lows[i] for i in range(2)], [ci_highs[i] - rates[i] for i in range(2)]]

bars = ax1.bar(groups, rates, yerr=yerr, capsize=8, color=["#4C72B0", "#55A868"],
               width=0.55, edgecolor="white", error_kw={"linewidth": 1.8, "ecolor": "#333333"})
ax1.set_ylabel("Conversion Rate (%)")
ax1.set_xlabel("Version")
ax1.set_title("Conversion Rates with 95% Confidence Intervals", fontweight="bold")
ax1.set_ylim(0, max(ci_highs) * 1.25)
for bar, rate, low, high in zip(bars, rates, ci_lows, ci_highs):
    ax1.text(bar.get_x() + bar.get_width()/2, high + 0.8,
             f"{rate:.1f}%\n[{low:.1f}–{high:.1f}]", ha="center", va="bottom", fontsize=10)

ax2 = axes[1]
diff_pct = abs_diff * 100
ci_diff_pct = (ci_diff[0] * 100, ci_diff[1] * 100)
ax2.axhline(0, color="gray", linestyle="--", linewidth=1)
ax2.errorbar([0], [diff_pct],
             yerr=[[diff_pct - ci_diff_pct[0]], [ci_diff_pct[1] - diff_pct]],
             fmt="o", markersize=10, color="#C44E52", capsize=10, linewidth=2)
ax2.set_xlim(-0.6, 0.6)
ax2.set_xticks([])
ax2.set_ylabel("Difference (B − A) in pp")
ax2.set_title("Estimated Lift with 95% CI", fontweight="bold")
ax2.text(0, ci_diff_pct[1] + 0.6, f"+{diff_pct:.1f} pp\n[{ci_diff_pct[0]:.1f}, {ci_diff_pct[1]:.1f}]",
         ha="center", va="bottom", fontsize=11, fontweight="bold")
ax2.axhspan(ci_diff_pct[0], ci_diff_pct[1], alpha=0.15, color="#C44E52")
if ci_diff_pct[0] < 0 < ci_diff_pct[1]:
    ax2.text(0.35, 0, "CI crosses zero\n→ not significant", fontsize=9, color="#555555", ha="left", va="center")

plt.tight_layout()
plt.savefig("ab_analysis_with_ci.png", dpi=160, bbox_inches="tight")
plt.show()
print("Figure saved: ab_analysis_with_ci.png")


## 7. Business Interpretation & Recommendations

### Statistical Summary
| Metric                              | Value                      |
|-------------------------------------|----------------------------|
| Version A conversion rate           | 14.0% (95% CI ≈ 10.2–18.8%) |
| Version B conversion rate           | 17.6% (95% CI ≈ 13.4–22.8%) |
| Absolute difference (B − A)         | +3.6 percentage points     |
| Relative lift                       | +25.7%                     |
| Two-proportion z-test p-value       | ≈ 0.27                     |
| 95% CI for difference               | includes zero              |
| Cohen’s h                           | ≈ 0.10 (small)             |
| Achieved power                      | ~20%                       |
| SRM test                            | No significant mismatch    |

### What this means for the business

1. **Directionally positive, but inconclusive** — CI for the difference includes zero.
2. **Underpowered experiment** — only ~20% power for the observed effect.
3. **Do not ship Version B solely on this evidence.**

### Recommended Actions
| Priority | Action |
|----------|--------|
| High | Do not fully roll out Version B |
| High | Re-run with larger sample (see power table) |
| Medium | Check secondary metrics and segments |
| Low | Consider sequential testing for future experiments |

**Next experiment design tip:** To detect a 10% relative lift (14% → 15.4%) with 80% power you need roughly 8,000–9,000 users per group.


## 8. Limitations & Assumptions

- **Data provenance unknown** — original source and collection process are not documented.
- Independence and fixed-sample design assumed.
- Binary outcome only; no revenue or retention metrics.
- Single primary comparison (no multiple-testing adjustment needed).


## 9. How to Cite This Work

> Folasunkanmi. (2026). *A/B Test Analysis: Conversion Rate Comparison* [Jupyter Notebook]. GitHub.  
> https://github.com/folasunkanmi/python-ab-testing

### Reproducibility checklist
- [x] Raw data provided
- [x] Strict validation with early stop
- [x] SRM test
- [x] Confidence intervals
- [x] Primary two-proportion z-test
- [x] Power analysis
- [x] Business recommendations
- [x] Automated tests + CI


In [ ]:
print("=" * 60)
print("Analysis complete.")
print("=" * 60)
